# 08 — Gold Layer: Business Metrics (Data Marts)
Calculates final KPIs by querying the Star Schema (fact_trips + dimensions).

In [ ]:
from src.core.spark import get_spark
from src.core.mongo import read_mongo
from src.core.config import settings
import pyspark.sql.functions as F

In [ ]:
spark = get_spark("TLC_Gold_Metrics")

In [ ]:
fact_trips = read_mongo(spark, "tlc_gold", "fact_trips")
dim_zone = read_mongo(spark, "tlc_gold", "dim_zone")
dim_vehicle = read_mongo(spark, "tlc_gold", "dim_vehicle")

In [ ]:
# KPI 1: Total Trips and Revenue by Vehicle Type
kpi1 = fact_trips.groupBy("vehicle_id").agg(
    F.sum("total_amount").alias("total_revenue"),
    F.count("*").alias("total_trips")
).join(
    dim_vehicle, 
    fact_trips["vehicle_id"] == dim_vehicle["id"], 
    "left"
).select(
    "name", "total_trips", "total_revenue"
)
kpi1.show(truncate=False)

In [ ]:
# KPI 2: Top 10 Pickup Zones by Revenue
kpi2 = fact_trips.groupBy("pickup_zone_id").agg(
    F.sum("total_amount").alias("total_revenue")
).join(
    dim_zone, 
    fact_trips["pickup_zone_id"] == dim_zone["zone_id"], 
    "left"
).select(
    "zone_name", "borough", "total_revenue"
).orderBy(
    F.col("total_revenue").desc()
).limit(10)
kpi2.show(truncate=False)

In [ ]:
# KPI 3: Congestion Fee Impact
kpi3 = fact_trips.groupBy("vehicle_id", "pickup_date_id").agg(
    F.sum("cbd_congestion_fee").alias("total_congestion_fee")
).orderBy(
    "pickup_date_id", "vehicle_id"
)
kpi3.show(truncate=False)